## Search Tree Assignment Template

Put a description of the game here along with any links to video or description of gameplay

This assignment must use the current board  and the search tree node as below



In [764]:
import math

# Game State

## Bitboard

Board states are stored minimally as the bits of a number. This approach minimizes the memory space of board states, and simplifies the use of mask operations through bitwise operations.

In [765]:
class BitBoard:
    def __init__(self, width, height, dataWidth, state = 0):
        self.width = width
        self.height = height
        self.dataWidth = dataWidth
        self.state = state
    
    def inBounds(self, x, y):
        return x >= 0 and x < self.width and y >= 0 and y < self.height
    
    def getXY(self, i):
        x = i % self.width
        y = self.height - 1 - (i // self.width)

        if(self.inBounds(x, y)):
            return x, y
        else:
            return -1, -1

    def getI(self, x, y):
        if(self.inBounds(x, y)):
            return (self.height - 1 - y) * self.width + x
        else:
            return -1
        
    def getAt(self, i):
        return (self.state >> (i * self.dataWidth)) & ((1 << self.dataWidth) - 1)
    
    def getAt2D(self, x, y):
        i = self.getI(x, y)
        if(i == -1):
            return None

        return self.getAt(i)

    def setAt(self, i, value):
        mask = ((1 << self.dataWidth) - 1) << i * self.dataWidth
        self.state &= ~mask
        self.state |= value << i * self.dataWidth

    def setAt2D(self, x, y, value):
        i = self.getI(x, y)
        if(i == -1):
            return

        return self.setAt(i, value)

### Chessboard (Mask)

A 1-bit-per-square representation of a chessboard. This class is used to generate masks showing the possible moves of pieces.

In [766]:
class ChessBoardMask(BitBoard):
    def __init__(self, state = 0):
        BitBoard.__init__(self, 8, 8, 1, state)
    
    def flipAt2D(self, x, y):
        super().setAt2D(x, y, 0 if self.getAt2D(x, y) else 1)

    def flipAt(self, i):
        super().setAt(i, 0 if self.getAt(i) else 1)
    
    def getAt(self, i):
        return True if super().getAt(i) == 1 else False
    
    def setAt(self, i, value):
        super().setAt(i, 1 if value else 0)
    
    def setAt2D(self, x, y, value):
        super().setAt2D(x, y, 1 if value else 0)
    
    def getAt2D(self, x, y):
        return True if super().getAt2D(x, y) == 1 else False

    def getOr(self, mask):
        return ChessBoardMask(self.state | mask.state)

    def getAnd(self, mask):
        return ChessBoardMask(self.state & mask.state)

    def getIncludes(self, mask):
        return (self.state & mask.state) == mask.state
    
    def validPositions(self):
        for y in range(self.height):
            for x in range(self.width):
                if(self.getAt2D(x,y)):
                    yield (x, y)

    def __str__(self):
        box_width = 3

        string = ""
        string += " " + " ".center(box_width) + " "
        string += "┌"
        for x in range(self.width - 1):
            string += "─" * box_width + "┬"
        string += "─" * box_width + "┐\n"

        for y in range(self.height):

            string += " "
            string += str(self.width - y).center(box_width) + " "

            string += "│"
            for x in range(self.width):
                value = self.getAt2D(x, self.height - y - 1)
                symbol = "X" if value else " "
                string += symbol.center(box_width) + "│"

            string += "\n " + " ".center(box_width) + " "

            if(y < self.height - 1):
                string += "├" + (("─" * box_width + "┼") * (self.width - 1)) + ("─" * box_width + "┤")
            else:
                string += "└" + (("─" * box_width + "┴") * (self.width - 1)) + ("─" * box_width + "┘")

            string += "\n"
        
        string += "  " + " ".center(box_width) + " "
        for letter in ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h']:
            string += letter.center(box_width) + " "

        return string

In [767]:
mask = ChessBoardMask()
mask.setAt2D(1,6, 1)
mask.setAt2D(6,6, 1)
mask.setAt2D(1,3, 1)
mask.setAt2D(2,2, 1)
mask.setAt2D(3,2, 1)
mask.setAt2D(4,2, 1)
mask.setAt2D(5,2, 1)
mask.setAt2D(6,3, 1)
print(mask)

     ┌───┬───┬───┬───┬───┬───┬───┬───┐
  8  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  7  │   │ X │   │   │   │   │ X │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  6  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  5  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  4  │   │ X │   │   │   │   │ X │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  3  │   │   │ X │ X │ X │ X │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  2  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  1  │   │   │   │   │   │   │   │   │
     └───┴───┴───┴───┴───┴───┴───┴───┘
       a   b   c   d   e   f   g   h  


## Moves

Chess pieces moves are defined by move definitions, with the following paremeters:

| Index | Parameter | Type | Description |
|-|-|-|-|
| 0 | Direction | (Int, Int) | The normalized direction of the move. Or, if the move is a jump, the number of squares to move on the x axis then the y axis. |
| 1 | Strength | Int | The maximum number of squares the move may move in Direction, or -1 if infinite. |
| 2 | Mirror X | Bool | If the move is mirrored on the X axis. E.g. (1,0) will also become (-1, 0) |
| 3 | Mirror Y | Bool | If the move is mirrored on the Y axis. E.g. (0,1) will also become (0,-1) |
| 4 | Jump | Bool | If the move can jump over pieces. This overwrites direction behaviour. |
| 5 | Can Capture | Bool | If the move may capture other pieces. |
| 6 | Must Capture | Bool | If the move must capture other pieces. E.g. a pawn's diagonal movement. |

A MoveDef returns all possible square a move may be able to visit, given an x and y position.

In [768]:
class MoveDef:
    def __init__(self, direction, strength = -1, mirrorX = False, mirrorY = False, jump = False, canCapture = False, mustCapture = False):
        self.boundsX = 8
        self.boundsY = 8
        
        self.direction = direction
        self.strength = strength
        self.mirrorX = mirrorX
        self.mirrorY = mirrorY
        self.jump = jump
        self.canCapture = canCapture
        self.mustCapture = mustCapture
    def getPossibleFrom(self, x, y, flipY = False):
        possible = {}

        if(not self.jump):
            directions = [(self.direction[0], self.direction[1] if not flipY else -1 * self.direction[1])]
            root_dir = directions[0]
            if(self.mirrorX):
                directions.append((root_dir[0] * -1, root_dir[1]))
            if(self.mirrorY):
                directions.append((root_dir[0], root_dir[1] * - 1))
            if(self.mirrorX and self.mirrorY):
                directions.append((root_dir[0] * -1, root_dir[1] * - 1))

            for direction in directions:
                possible[direction] = []
                # This little ternary (and only it) removes support of asymmetrical boards
                strengthI = self.boundsX if self.strength <= 0 else self.strength

                for i in range(1, strengthI + 1):
                    curX = x + (direction[0] * i)
                    curY = y + (direction[1] * i)
                    if(not self.inBounds(curX, curY)):
                        break
                    possible[direction].append((curX, curY))
        elif(self.jump):
            directions = [(self.direction[0], self.direction[1] if not flipY else -1 * self.direction[1])]
            root_dir = directions[0]
            if(self.mirrorX):
                directions.append((root_dir[0] * -1, root_dir[1]))
            if(self.mirrorY):
                directions.append((root_dir[0], root_dir[1] * - 1))
            if(self.mirrorX and self.mirrorY):
                directions.append((root_dir[0] * -1, root_dir[1] * - 1))

            for direction in directions:
                possible[direction] = []
                pos = (x + direction[0], y + direction[1])
                if(self.inBounds(pos[0], pos[1])):
                    possible[direction].append(pos)

        return possible
    def inBounds(self, x, y):
        return x >= 0 and x < self.boundsX and y >= 0 and y < self.boundsY


## Pieces

Each chessboard is a 256 bit (32 byte) number, where each cell is represented by 1 bit representing color (0 for white, 1 for black) and 3 bits representing piece-type. E.g. `0b1001` for a black pawn, `0b0010` for a white knight, etc. 

| Piece | Type Bits | White | Black |
|-|-|-|-|
| Empty | `000` | `0b0000` | `0b1000` |
| Pawn | `001` | `0b0001` | `0b1001` |
| Knight | `010` | `0b0010` | `0b1010` |
| Bishop | `011` | `0b0011` | `0b1011` |
| Rook | `100` | `0b0100` | `0b1100` |
| Queen | `101` | `0b0101` | `0b1101` |
| King | `110` | `0b0110` | `0b1110` |

Pieces use MoveDefs to calculate a mask of all possible moves they can make, accounting for blocked squares and captures.

In [769]:
class Piece:
    WHITE_VALUE = 0b0000
    BLACK_VALUE = 0b1000
    
    @staticmethod
    def isWhite(coloredPieceValue):
        return Piece.extractColor(coloredPieceValue) == 0
    
    @staticmethod
    def extractColor(coloredPieceValue):
        return (coloredPieceValue & 0b1000)

    @staticmethod
    def extractPiece(coloredPieceValue):
        return (coloredPieceValue & 0b0111)

    def __init__(self, name, symbols, value, moveDefinitions):
        self.name = name
        self.symbols = symbols
        self.value = value
        self.moveDefinitions = moveDefinitions

    def __str__(self):
        return self.name

    def getValidMoves(self, state, position, onlyCapturing = False):
        isWhite = Piece.isWhite(state.getAt(position))
        x, y = state.getXY(position)

        if(x == -1 or y == -1):
            raise Exception("Attempted to solve moves for piece at invalid position index " + position + ".")

        mask = ChessBoardMask()

        for move in self.moveDefinitions:
            if(onlyCapturing and not move.canCapture):
                continue

            possibles = move.getPossibleFrom(x, y, not isWhite)
            
            for direction in possibles:
                positions = possibles[direction]
                for position in positions:
                    value = state.getAt2D(position[0], position[1])

                    if(PieceDefs.isEmpty(value) and not move.mustCapture):
                        mask.setAt2D(position[0], position[1], True)
                    elif(not PieceDefs.isEmpty(value)):
                        if(move.canCapture and (Piece.extractColor(value) != (Piece.WHITE_VALUE if isWhite else Piece.BLACK_VALUE))):
                            mask.setAt2D(position[0], position[1], True)
                        if(not move.jump): # Because custom pieces aren't neeeded, 'jump' also messes with move generation
                            break # Cannot move over piece
        return mask
    def getSymbol(self, asWhite):
        return self.symbols[0 if asWhite else 1]
    def getValue(self, asWhite):
        return self.value | (Piece.WHITE_VALUE if asWhite else Piece.BLACK_VALUE)
    def isValue(self, value):
        return (value & 0b0111) == self.value

class PieceDefs:
    EMPTY_VALUE = 0b000
    PAWN_VALUE = 0b001
    KNIGHT_VALUE = 0b010
    BISHOP_VALUE = 0b011
    ROOK_VALUE = 0b100
    QUEEN_VALUE = 0b101
    KING_VALUE = 0b110

    EMPTY = Piece("", ("  ", "  "), EMPTY_VALUE, [])
    PAWN = Piece("Pawn", ("P", "p"), PAWN_VALUE, [
        MoveDef((0,1), 1, False, False, False, False, False),
        MoveDef((1,1), 1, True, False, False, True, True),
    ])
    KNIGHT = Piece("Knight", ("N", "n"), KNIGHT_VALUE, [
        MoveDef((2, 1), -1, True, True, True, True, False)
    ])
    BISHOP = Piece("Bishop", ("B", "b"), BISHOP_VALUE, [
        MoveDef((1,1), -1, True, True, False, True, False)
    ])
    ROOK = Piece("Rook", ("R", "r"), ROOK_VALUE, [
        MoveDef((1,0), -1, True, False, False, True, False),
        MoveDef((0,1), -1, False, True, False, True, False)
    ])
    QUEEN = Piece("Queen", ("Q", "q"), QUEEN_VALUE, [
        MoveDef((1,0), -1, True, False, False, True, False),
        MoveDef((0,1), -1, False, True, False, True, False),
        MoveDef((1,1), -1, True, True, False, True, False)
    ])
    KING = Piece("King", ("K", "k"), KING_VALUE, [
        MoveDef((1,0), 1, True, False, False, True, False),
        MoveDef((0,1), 1, False, True, False, True, False),
        MoveDef((1,1), 1, True, True, False, True, False)
    ])
    
    VALUE_MAP = {
        EMPTY_VALUE: EMPTY,
        PAWN_VALUE: PAWN,
        KNIGHT_VALUE: KNIGHT,
        BISHOP_VALUE: BISHOP,
        ROOK_VALUE: ROOK,
        QUEEN_VALUE : QUEEN,
        KING_VALUE : KING
    }

    @staticmethod
    def isEmpty(value):
        return value == PieceDefs.EMPTY.getValue(True) or value == PieceDefs.EMPTY.getValue(False)

## Chessboard

The chessboard state is tracked as a 256 bit number, made up of pieces and their colours.

In [770]:
class ChessBoardState(BitBoard):
    def __init__(self, state = 0):
        BitBoard.__init__(self, 8, 8, 4, state)
        self.advantage = 0
    
    def copyWithMove(self, x, y, newX, newY):
        copy = ChessBoardState(self.state)
        val = copy.getAt2D(x, y)
        copy.setAt2D(x, y, PieceDefs.EMPTY.getValue(True))
        copy.setAt2D(newX, newY, val)
        return copy
    # Get all pieces on a board, or get all pieces on a board whose colour bit equals filter.
    def getPieces(self, filter = None):
        pieces = [] # (x, y), piece
        for i in range(self.width * self.height):
            value = self.getAt(i)
            if(not PieceDefs.isEmpty(value)):
                if(filter is None or Piece.extractColor(value) == filter):
                    pieces.append((self.getXY(i), value))
        return pieces
    def getKingPosition(self, white):
        kingPosition = None
        for position, piece in self.getPieces(Piece.WHITE_VALUE if white else Piece.BLACK_VALUE):
            if(piece == PieceDefs.KING.getValue(white)):
                kingPosition = position
                break
        return kingPosition
    # Get all possible moves on a board.
    def getMoves(self, onlyCapturing = False, filter = None):
        moves = [] # (x, y), piece, mask

        pieces = self.getPieces(filter)

        for position, piece in pieces:
            moves.append((position, piece, PieceDefs.VALUE_MAP[Piece.extractPiece(piece)].getValidMoves(self, self.getI(position[0], position[1]), onlyCapturing)))

        return moves
    def __str__(self):
        box_width = 3

        string = ""
        string += " " + " ".center(box_width) + " "
        string += "┌"
        for x in range(self.width - 1):
            string += "─" * box_width + "┬"
        string += "─" * box_width + "┐\n"

        for y in range(self.height):

            string += " "
            string += str(self.width - y).center(box_width) + " "

            string += "│"
            for x in range(self.width):
                pieceValue = self.getAt2D(x, self.height - y - 1)
                piece = PieceDefs.VALUE_MAP[Piece.extractPiece(pieceValue)]
                symbol = piece.getSymbol(Piece.isWhite(pieceValue))
                string += symbol.center(box_width) + "│"

            string += "\n " + " ".center(box_width) + " "

            if(y < self.height - 1):
                string += "├" + (("─" * box_width + "┼") * (self.width - 1)) + ("─" * box_width + "┤")
            else:
                string += "└" + (("─" * box_width + "┴") * (self.width - 1)) + ("─" * box_width + "┘")

            string += "\n"
        
        string += "  " + " ".center(box_width) + " "
        for letter in ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h']:
            string += letter.center(box_width) + " "
        string += "\n" + (" " * round((box_width * (self.width / 2)))) + "Advantage: " + str(self.advantage)

        return string

### Defining a board with pieces

In [771]:
board = ChessBoardState()

board.setAt2D(2, 2, PieceDefs.PAWN.getValue(True))
board.setAt2D(2, 1, PieceDefs.KNIGHT.getValue(True))
board.setAt2D(3, 3, PieceDefs.BISHOP.getValue(False))

print(board)

     ┌───┬───┬───┬───┬───┬───┬───┬───┐
  8  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  7  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  6  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  5  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  4  │   │   │   │ b │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  3  │   │   │ P │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  2  │   │   │ N │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  1  │   │   │   │   │   │   │   │   │
     └───┴───┴───┴───┴───┴───┴───┴───┘
       a   b   c   d   e   f   g   h  
            Advantage: 0


### Finding valid moves
The pawn at c3 is capable of moving forward one square, or capturing on d4. It is not able to move b4 as there is no piece to capture.

In [772]:
print(PieceDefs.PAWN.getValidMoves(board, board.getI(2, 2), False))

     ┌───┬───┬───┬───┬───┬───┬───┬───┐
  8  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  7  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  6  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  5  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  4  │   │   │ X │ X │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  3  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  2  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  1  │   │   │   │   │   │   │   │   │
     └───┴───┴───┴───┴───┴───┴───┴───┘
       a   b   c   d   e   f   g   h  


### Updating the board

In [773]:
board = board.copyWithMove(2,2, 3, 3)
print(board)

     ┌───┬───┬───┬───┬───┬───┬───┬───┐
  8  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  7  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  6  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  5  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  4  │   │   │   │ P │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  3  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  2  │   │   │ N │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  1  │   │   │   │   │   │   │   │   │
     └───┴───┴───┴───┴───┴───┴───┴───┘
       a   b   c   d   e   f   g   h  
            Advantage: 0


# State Evaluation

## Check and Checkmate

In [774]:
# Get all check sources targeting kingPosition, and return a map of all squares attacked by the opposite colour.
def getChecks(board, kingPosition):
    isWhite = Piece.isWhite(board.getAt2D(kingPosition[0], kingPosition[1]))

    moves = board.getMoves(True, Piece.BLACK_VALUE if isWhite else Piece.WHITE_VALUE)

    checkSources = []

    attackedMask = ChessBoardMask()
    for position, piece, moves in moves:
        if(moves.getAt2D(kingPosition[0], kingPosition[1])):
            checkSources.append((position, piece, moves))

        attackedMask = attackedMask.getOr(moves)

    return (checkSources, attackedMask)

# Search-Tree Expansion (Min-Max)

In [775]:
def expand(board, asWhite, advantageFunction = None, advantageModifier = 0):
    moves = board.getMoves(filter = (Piece.WHITE_VALUE if asWhite else Piece.BLACK_VALUE))

    states = []
    for sourcePosition, piece, mask in moves:
        for targetPosition in mask.validPositions():
            newBoard = board.copyWithMove(sourcePosition[0], sourcePosition[1], targetPosition[0], targetPosition[1])
            
            kingPosition = newBoard.getKingPosition(asWhite)
            newIsChecked = False
            if kingPosition is not None:
                checkSources, _ = getChecks(newBoard, kingPosition)
                newIsChecked = len(checkSources) > 0

            if(newIsChecked):
                continue # Moving into checked position, illegal.
            
            if(advantageFunction is not None):
                newBoard.advantage = advantageFunction(newBoard, advantageModifier) 
            else:
                newBoard.advantage = advantageModifier
            states.append(newBoard)

    return states

In [776]:
def isCheckmated(board, asWhite):
    kingPosition = board.getKingPosition(asWhite)
    if(kingPosition is None):
        return False
    
    checkSources, _ = getChecks(board, kingPosition)
    if(len(checkSources) <= 0):
        return False
    
    return len(expand(board, asWhite)) <= 0

def isStalemated(board, asWhite):
    return len(expand(board, asWhite)) <= 0

In [777]:
# https://www.geeksforgeeks.org/artificial-intelligence/mini-max-algorithm-in-artificial-intelligence/
def minmax(board, depth, asWhite, advantageFunction):
    if(isCheckmated(board, asWhite) or depth == 0):
        return board.advantage

    states = expand(board, asWhite, advantageFunction, depth)
    
    if(not states or len(states) <= 0):
        return board.advantage

    if asWhite:
        max_eval = float('-inf')
        for state in states:
            eval = minmax(state, depth - 1, not asWhite, advantageFunction)
            max_eval = max(max_eval, eval)
        return max_eval
    else:
        max_eval = float('inf')
        for state in states:
            eval = minmax(state, depth - 1, not asWhite, advantageFunction)
            max_eval = min(max_eval, eval)
        return max_eval

def bestMove(board, depth, asWhite, advantageFunction):
    states = expand(board, asWhite, advantageFunction)
    if(not states or len(states) <= 0):
        return board
    
    best_eval = ChessBoardState()
    best_eval.advantage = float('-inf') if asWhite else float('inf')  

    if asWhite: 
        for state in states:
            eval = minmax(state, depth - 1, not asWhite, advantageFunction)
            if(eval > best_eval.advantage):
                best_eval = state
    else:
        for state in states:
            eval = minmax(state, depth - 1, not asWhite, advantageFunction)
            if(eval < best_eval.advantage):
                best_eval = state
    
    return best_eval

Expanding the previous example board, the pawn moves forward one square.

In [778]:
print(expand(board, True)[0])

     ┌───┬───┬───┬───┬───┬───┬───┬───┐
  8  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  7  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  6  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  5  │   │   │   │ P │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  4  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  3  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  2  │   │   │ N │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  1  │   │   │   │   │   │   │   │   │
     └───┴───┴───┴───┴───┴───┴───┴───┘
       a   b   c   d   e   f   g   h  
            Advantage: 0


## Advantage

Advantage methods always assume the computer plays first (a.k.a. the computer is white).

### Version 1 - Simple 'Have I won?'

Before implementing more advanced methods of evaluating board advantage, an expanded search tree will be used to identify a simple checkmate in one or two moves.

In [779]:
def advantage_checkmate(board, depth):
    if(isCheckmated(board, False)):
        return 1
    elif(isStalemated(board, False)):
        return -1
    else:
        return 0

#### Testing: Find a Mate In 2

In [780]:
board = ChessBoardState()
board.setAt2D(0,0, PieceDefs.KING.getValue(True))
board.setAt2D(7,7, PieceDefs.KING.getValue(False))
board.setAt2D(6,0, PieceDefs.ROOK.getValue(True))
board.setAt2D(5,0, PieceDefs.ROOK.getValue(True))

print(board)

     ┌───┬───┬───┬───┬───┬───┬───┬───┐
  8  │   │   │   │   │   │   │   │ k │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  7  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  6  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  5  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  4  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  3  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  2  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  1  │ K │   │   │   │   │ R │ R │   │
     └───┴───┴───┴───┴───┴───┴───┴───┘
       a   b   c   d   e   f   g   h  
            Advantage: 0


In [781]:
best = bestMove(board, 3, True, advantage_checkmate)
print(best)
best = bestMove(best, 3, False, advantage_checkmate)
print(best)
best = bestMove(best, 3, True, advantage_checkmate)
print(best)

     ┌───┬───┬───┬───┬───┬───┬───┬───┐
  8  │   │   │   │   │   │   │   │ k │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  7  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  6  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  5  │   │   │   │   │   │   │ R │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  4  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  3  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  2  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  1  │ K │   │   │   │   │ R │   │   │
     └───┴───┴───┴───┴───┴───┴───┴───┘
       a   b   c   d   e   f   g   h  
            Advantage: 0
     ┌───┬───┬───┬───┬───┬───┬───┬───┐
  8  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  7  │   │   │   │   │   │   │   │ k │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  6  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤


### Version 2 - King + Queen Endgame

There are two simple scores that can quantified to determine closeness to winning in a King Queen endgame:

* How close is the enemy king to the centre of the board? Further is better.
* How close is the king to the enemy king? Closer is better.

With the immediate losing conditions of:

* No Queen (draw).
* Stalemate.

In [782]:
def distance(x1, y1, x2, y2):
    return math.sqrt(((x2 - x1) ** 2) + ((y2 - y1) ** 2))

def advantage_kingqueen(board, depth):
    if(isCheckmated(board, False)):
        return 99
    elif(isStalemated(board, False)):
        return -99
    
    kingPos = None
    queenPos = None
    enemyKingPos = None
    
    for pos, piece in board.getPieces():
        if(piece == PieceDefs.KING.getValue(True)):
            kingPos = pos
        elif(piece == PieceDefs.QUEEN.getValue(True)):
            queenPos = pos
        elif(piece == PieceDefs.KING.getValue(False)):
            enemyKingPos = pos

    if(queenPos == None):
        return -1
    
    fromCenterDist = distance(board.width / 2, board.height / 2,enemyKingPos[0], enemyKingPos[1])
    kingsDist = distance(kingPos[0], kingPos[1], enemyKingPos[0], enemyKingPos[1])

    return fromCenterDist - kingsDist

#### Testing

In [783]:
board = ChessBoardState()
board.setAt2D(0,0, PieceDefs.KING.getValue(True))
board.setAt2D(3,3, PieceDefs.KING.getValue(False))
board.setAt2D(1,0, PieceDefs.QUEEN.getValue(True))

print(board)

     ┌───┬───┬───┬───┬───┬───┬───┬───┐
  8  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  7  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  6  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  5  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  4  │   │   │   │ k │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  3  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  2  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  1  │ K │ Q │   │   │   │   │   │   │
     └───┴───┴───┴───┴───┴───┴───┴───┘
       a   b   c   d   e   f   g   h  
            Advantage: 0


In [784]:
""" // 1 min~ runtime
best = bestMove(board, 3, True, advantage_kingqueen)
move = True
while(best.advantage < 99):
    move = not move
    best = bestMove(best, 3, move, advantage_kingqueen)
print(best)
"""

' // 1 min~ runtime\nbest = bestMove(board, 3, True, advantage_kingqueen)\nmove = True\nwhile(best.advantage < 99):\n    move = not move\n    best = bestMove(best, 3, move, advantage_kingqueen)\nprint(best)\n'

#### Incorporating the Queen

It works! But the performance leaves a lot to be desired. This is because the current implementation only cares about the kings' positions, and doesn't properly utilize the queen to close out the game quickly. To try encourage this, the queen's distance to the enemy king can be evaluated. The closer it is, the more squares it will restrict.

Careful weighting is required here. The queen's positioning is not nearly as important as the enemy king's distance from the centre, and the allied king MUST be next to the enemy king to find a checkmate. This version is also informed about the depth, making sure to pick the fastest checkmate.

In [785]:
def advantage_kingqueen(board, depth):
    if(isCheckmated(board, False)):
        return 99
    elif(isStalemated(board, False)):
        return -99
    
    kingPos = None
    queenPos = None
    enemyKingPos = None
    
    for pos, piece in board.getPieces():
        if(piece == PieceDefs.KING.getValue(True)):
            kingPos = pos
        elif(piece == PieceDefs.QUEEN.getValue(True)):
            queenPos = pos
        elif(piece == PieceDefs.KING.getValue(False)):
            enemyKingPos = pos

    if(queenPos == None):
        return -1
    
    fromCenterDist = distance(board.width / 2, board.height / 2,enemyKingPos[0], enemyKingPos[1])
    kingsDist = distance(kingPos[0], kingPos[1], enemyKingPos[0], enemyKingPos[1])
    kingQueenDist = distance(queenPos[0], queenPos[1], enemyKingPos[0], enemyKingPos[1])

    return (fromCenterDist * 2) - (kingsDist * 3) - kingQueenDist - (depth * 0.1)

#### Testing

In [786]:
""" 13 sec~ runtime
board = ChessBoardState()
board.setAt2D(0,0, PieceDefs.KING.getValue(True))
board.setAt2D(3,3, PieceDefs.KING.getValue(False))
board.setAt2D(1,0, PieceDefs.QUEEN.getValue(True))

best = bestMove(board, 3, True, advantage_kingqueen)
move = True
while(best.advantage < 99):
    move = not move
    best = bestMove(best, 3, move, advantage_kingqueen)
print(best)
"""

' 13 sec~ runtime\nboard = ChessBoardState()\nboard.setAt2D(0,0, PieceDefs.KING.getValue(True))\nboard.setAt2D(3,3, PieceDefs.KING.getValue(False))\nboard.setAt2D(1,0, PieceDefs.QUEEN.getValue(True))\n\nbest = bestMove(board, 3, True, advantage_kingqueen)\nmove = True\nwhile(best.advantage < 99):\n    move = not move\n    best = bestMove(best, 3, move, advantage_kingqueen)\nprint(best)\n'

With this new strategy, the algorithm runs much faster. The same checkmate can even be found with a reduced depth:

In [787]:
board = ChessBoardState()
board.setAt2D(0,0, PieceDefs.KING.getValue(True))
board.setAt2D(3,3, PieceDefs.KING.getValue(False))
board.setAt2D(1,0, PieceDefs.QUEEN.getValue(True))

best = bestMove(board, 2, True, advantage_kingqueen)
move = True
while(best.advantage < 99):
    move = not move
    best = bestMove(best, 2, move, advantage_kingqueen)
print(best)

     ┌───┬───┬───┬───┬───┬───┬───┬───┐
  8  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  7  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  6  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  5  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  4  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  3  │   │   │ K │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  2  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  1  │   │   │   │ k │   │ Q │   │   │
     └───┴───┴───┴───┴───┴───┴───┴───┘
       a   b   c   d   e   f   g   h  
            Advantage: 99


## CurrentBoard

Integrating the logic defined above into the template includes the following modifications:

* all_possible_moves automatically evaluates board states as they're built.
* state_of_board returns a tuple of two bools. The first indicates stalemate, the second indicates checkmate. There is no draw detection built in.

In [788]:
class CurrentBoard:
  def __init__(self, board_def = 0) :
    self.board = ChessBoardState(board_def)

  def display(self):
    print(self.board)

  def state_of_board(self, isWhite):
    return (isStalemated(self.board, isWhite), isCheckmated(self.board, isWhite))

  def all_possible_moves(self, isWhite, advantageFunction, advantageModifier = 0):
    return [CurrentBoard(board.state) for board in expand(self.board, isWhite, advantageFunction, advantageModifier)]

  def evaluation(self):
    return self.board.advantage

## Search Tree Node

There may be a need to adjust this code to deal with that trees are too large.

This is done by adding a max ply depth variable and only generating children for search tree nodes whose ply depth it is less than this max ply depth.

This will require the evaluation function discussed above to be applied to the nodes at this max depth.

In [789]:
class SearchTreeNode:

  def __init__(self,board_instance, isWhite, advantageFunction, ply=0):
    self.children = []
    self.ply_depth = ply
    self.current_board = board_instance
    self.isWhite = isWhite
    self.advantageFunction = advantageFunction

    self.won = self.current_board.state_of_board(not self.isWhite)[0]
    self.lost = self.current_board.state_of_board(self.isWhite)[0]

  def min_max_value(self):
    return minmax(self.current_board.board, self.ply_depth, self.isWhite, self.advantageFunction)
  
  def best_move(self):
    best = bestMove(self.current_board.board, self.ply_depth, self.isWhite, self.advantageFunction)
    board = CurrentBoard(best.state)
    return SearchTreeNode(board, not self.isWhite, self.advantageFunction, self.ply_depth)


### Quick tests

In [790]:
board = ChessBoardState()
board.setAt2D(0,0, PieceDefs.KING.getValue(True))
board.setAt2D(3,3, PieceDefs.KING.getValue(False))
board.setAt2D(1,0, PieceDefs.QUEEN.getValue(True))

curBoard = CurrentBoard(board.state)
curBoard.display()

     ┌───┬───┬───┬───┬───┬───┬───┬───┐
  8  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  7  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  6  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  5  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  4  │   │   │   │ k │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  3  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  2  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  1  │ K │ Q │   │   │   │   │   │   │
     └───┴───┴───┴───┴───┴───┴───┴───┘
       a   b   c   d   e   f   g   h  
            Advantage: 0


In [791]:
best = SearchTreeNode(curBoard, True, advantage_kingqueen, 2)

move = True
while(not best.won):
    best = best.best_move()

best.current_board.display()

     ┌───┬───┬───┬───┬───┬───┬───┬───┐
  8  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  7  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  6  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  5  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  4  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  3  │   │   │ K │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  2  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  1  │   │   │   │ k │   │ Q │   │   │
     └───┴───┴───┴───┴───┴───┴───┴───┘
       a   b   c   d   e   f   g   h  
            Advantage: 0


##  Gameplay

Most likely that this method will have to be adjusted for your game.

In [792]:
from IPython.display import clear_output

def getPieceMoves(board, x, y, isWhite):
  moves = board.getMoves(Piece.WHITE_VALUE if isWhite else Piece.BLACK_VALUE)
  for pos, piece, mask in moves:
    if(pos == (x, y)):
      return mask
  return None

# "e2 e4" into (4,1), (4,3)
def fromNotation(s):
    cols = {'a':0,'b':1,'c':2,'d':3,'e':4,'f':5,'g':6,'h':7}
    src, dst = s.lower().strip().split()
    return (cols[src[0]], int(src[1])-1), (cols[dst[0]], int(dst[1])-1)

def play_chess(startingBoard, aiAdvantageFunction):
  response = input("Play as White? (y/n): ")
  playersTurn = (response.lower() == "y")
  whiteMove = True

  currentBoard = startingBoard
  

  while True:
    clear_output(wait=True)
    currentBoard.display()

    stalemated, checkmated = currentBoard.state_of_board(whiteMove)
    if checkmated:
      if playersTurn:
        print("Checkmate! You lose.")
      else:
        print("Checkmate! You win!")
      break
    elif stalemated:
      print("Stalemate! Draw.")
      break

    if playersTurn:
      print("Your move.")
      
      while True:
        try:
          notationStr = input("Move [FROM] [TO] (e.g. E4 E3): ")
          (fromX, fromY), (toX, toY) = fromNotation(notationStr)

          mask = getPieceMoves(currentBoard.board, fromX, fromY, whiteMove)

          if mask is None:
            print("No piece there.")
            continue

          if not mask.getAt2D(toX, toY):
              print("Illegal move.")
              continue
          
          newBoard = currentBoard.board.copyWithMove(fromX, fromY, toX, toY)
          
          kingPosition = newBoard.getKingPosition(whiteMove)
          checkSources, _ = getChecks(newBoard, kingPosition)

          if(len(checkSources) > 0):
              print("Illegal move. Cannot move into check.")
              continue
          
          currentBoard.board = newBoard
          break
        except KeyboardInterrupt:
          print("Quitting...")
          return
        except Exception:
          print("Invalid Input")
    else:
      print("AI move...")
      node = SearchTreeNode(currentBoard, whiteMove, aiAdvantageFunction, 3)
      currentBoard = node.best_move().current_board

    whiteMove = not whiteMove
    playersTurn = not playersTurn


In [793]:
_gameBoard = ChessBoardState()
_gameBoard.setAt2D(0,0, PieceDefs.KING.getValue(True))
_gameBoard.setAt2D(3,3, PieceDefs.KING.getValue(False))
_gameBoard.setAt2D(1,0, PieceDefs.QUEEN.getValue(True))

gameBoard = CurrentBoard(_gameBoard.state)

play_chess(gameBoard, advantage_kingqueen)

     ┌───┬───┬───┬───┬───┬───┬───┬───┐
  8  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  7  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  6  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  5  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  4  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  3  │   │   │ K │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  2  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  1  │   │   │   │ k │   │ Q │   │   │
     └───┴───┴───┴───┴───┴───┴───┴───┘
       a   b   c   d   e   f   g   h  
            Advantage: 0
Checkmate! You lose.


## Testing and evaluation

Show that the algorithm is working, and evaluate its performance. Discuss any shortcomings and propose alterations to the evaluation function to address these. Repeat this evaluation and alteration highlighting the improvements, if any, the alterations make.

During this iterative process, do not overwrite code or results, If necessary cutting and pasting multiple versions of current board and search tree node into cells below so that the evolution of the assignment can be read from top to bottom.